# Day 07 - Creating New Columns (feature engineering)  (SOLUTIONS)

**Goal:** Build new columns with math, conditions, mapping and custom functions.

**Quick recap (read for 2 minutes):**

- Math: `df['new'] = df['a'] / df['b']`
- `np.where(condition, value_if_true, value_if_false)`
- `pd.cut(col, bins, labels)` fixed ranges; `pd.qcut(col, 4)` equal-sized groups
- `col.map(dict)` and `col.apply(function_or_lambda)`
- `df.assign(new1=..., new2=...)` creates several columns in one chain

*There are many valid ways to solve each question. If your answer gives the same result, it is correct.*

### Warm-up (5 min)
Without looking, redo Questions 1 to 3 from **Day 06** in a scratch cell below.

### Setup (run this first)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("spotify_user_behv.csv")

### Q1. Add `hours_per_day` = weekly listening hours / 7, rounded to 2 decimals.

In [ ]:
df["hours_per_day"] = (df["avg_listening_hours_per_week"] / 7).round(2)
df[["avg_listening_hours_per_week", "hours_per_day"]].head()

### Q2. Add `is_heavy_listener`: 'Yes' if weekly hours > 12 else 'No' (use `np.where`).

In [ ]:
df["is_heavy_listener"] = np.where(df["avg_listening_hours_per_week"] > 12, "Yes", "No")
df["is_heavy_listener"].value_counts()

### Q3. Create `age_group` with bins 16-24, 25-34, 35-44, 45-60 using `pd.cut`.

In [ ]:
df["age_group"] = pd.cut(df["age"], bins=[15, 24, 34, 44, 60], labels=["16-24", "25-34", "35-44", "45-60"])
df["age_group"].value_counts().sort_index()

### Q4. Create `plan_group`: 'Free' for Free users, 'Paid' for everyone else.

In [ ]:
df["plan_group"] = np.where(df["subscription_type"] == "Free", "Free", "Paid")
df["plan_group"].value_counts()

### Q5. Create `device_code` by mapping Mobile->M, Desktop->D, Tablet->T, Smart Speaker->S, Car System->C.

In [ ]:
df["device_code"] = df["primary_device"].map({"Mobile": "M", "Desktop": "D", "Tablet": "T", "Smart Speaker": "S", "Car System": "C"})
df["device_code"].value_counts()

### Q6. Using `apply` with a `lambda`, create `skip_level`: 'High' if skips > 12 else 'Normal'.

In [ ]:
df["skip_level"] = df["avg_skips_per_day"].apply(lambda x: "High" if x > 12 else "Normal")
df["skip_level"].value_counts()

### Q7. Write a function `listener_type(h)` returning 'Light' (<7), 'Medium' (7 to 13) or 'Heavy' (>13). Apply it to listening hours as `listener_type`.

In [ ]:
def listener_type(h):
    if h < 7:
        return "Light"
    elif h <= 13:
        return "Medium"
    else:
        return "Heavy"

df["listener_type"] = df["avg_listening_hours_per_week"].apply(listener_type)
df["listener_type"].value_counts()

### Q8. Create `engagement_score` = hours*0.5 + playlists*0.3 - skips*0.2, rounded to 2 decimals.

In [ ]:
df["engagement_score"] = (df["avg_listening_hours_per_week"] * 0.5 + df["playlists_created"] * 0.3 - df["avg_skips_per_day"] * 0.2).round(2)
df["engagement_score"].describe()

### Q9. Split users into 4 equal-sized groups (Q1 to Q4) by listening hours using `pd.qcut`.

In [ ]:
df["hours_quartile"] = pd.qcut(df["avg_listening_hours_per_week"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
df["hours_quartile"].value_counts().sort_index()

### Q10. Use `assign()` to create `hours_per_day` and `skips_per_week` in one step, saved as `df2`.

In [ ]:
df2 = df.assign(hours_per_day=df["avg_listening_hours_per_week"] / 7, skips_per_week=df["avg_skips_per_day"] * 7)
df2[["hours_per_day", "skips_per_week"]].head()